# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading, exploring, and processing the FAIR² Mental Health Survey Kilifi dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL based on the FAIR² standard.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment below if running the first time)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We start by importing the required modules and loading the dataset metadata from the Croissant JSON-LD URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Accessing and printing out high-level metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Cite as: {metadata.citeAs}")

## 2. Data Overview
Let's list all record sets, their fields, and their unique `@id` identifiers using the metadata. These `@id`s are essential when working programmatically with the Croissant structure and keep references consistent.

Below, we iterate over all record sets and display their `@id` and available fields.

In [ ]:
# List all record sets in the dataset
print("# Record sets found in this dataset:")
record_sets = dataset.record_sets
for rec in record_sets:
    print(f"  RecordSet name: {rec.name} | @id: {rec.id}")
    print("    Fields (field name | @id):")
    for field in rec.fields:
        print(f"      - {field.name} | {field.id}")
    print()

## 3. Data Extraction
Now, we'll extract the data from the primary record set (usually the core data table), loading it into a pandas DataFrame for convenient analysis.

Below, we identify the main record set, extract all its records, and show the first few rows. **All references use the record set and field `@id` values discovered above.**

In [ ]:
# Choose the primary record set (update if your dataset designates more than one core table)
# You may select the RecordSet @id from the overview step above.
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For demonstration, show field names of the first (main) record set
main_rs_id = record_set_ids[0]
print(f"Columns in record set ({main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
# Show the first records
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Here, perform basic filtering and normalization on one of the numeric fields from the main record set. All field references use their `@id`s.

- Filter records for a numeric field (e.g., GAD-7, PHQ-9, or PSQ score) where the value exceeds a given threshold.
- Normalize the selected field.
- Optionally, group by a categorical field (e.g., education level, gender).

Below, we demonstrate using the `gad7_score` field, adjusting as needed for your dataset.

In [ ]:
# Find candidate numeric field and group field by name/ID from metadata overview above
# Example fields (replace with exact @id as found in your dataset):
numeric_field_id = None
group_field_id = None
for f in dataset.record_sets[0].fields:
    # Find a likely numeric score field. Adjust as needed if field IDs differ.
    if f.name and ("gad" in f.name.lower() or "phq" in f.name.lower() or "psq" in f.name.lower()) and f.data_type == 'Float':
        numeric_field_id = f.id
    if f.name and ("education" in f.name.lower() or "gender" in f.name.lower()):
        group_field_id = f.id
# Sanity check
print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group-by field @id: {group_field_id}")

df = dataframes[main_rs_id]
if numeric_field_id in df.columns:
    # Drop missing values to avoid errors
    score_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = score_values.mean() if pd.notnull(score_values.mean()) else 10
    filtered_df = df[score_values > threshold].copy()

    print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (score_values - score_values.mean()) / score_values.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)
else:
    print(f"Numeric field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of the selected numeric mental health score (e.g., GAD-7) and its relation to a categorical variable such as education or gender, when available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram/distribution if the numeric field exists
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} Score by {group_field_id}")
        plt.show()
else:
    print(f"Field {numeric_field_id} not available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. 

We programmatically identified record sets and fields using their `@id`s, extracted the data, performed basic filtering and normalization on a mental health score field, and visualized the result.

This approach demonstrates how Croissant metadata and `mlcroissant` enable FAIR, interoperable data processing and analysis in real-world, multi-field research datasets.

*End of notebook.*